# 3D Scene to 2D Image: Camera Projection

**Goal:** understand how a 3D world point ends up as a pixel in a camera image.

This notebook follows directly from the *Practical Topics in Spatial AI* lecture on image formation and the camera calibration matrix.

By the end, you should be able to:

1. Compute the focal length in pixels from physical sensor and lens parameters.
2. Build the calibration matrix **K** from scratch.
3. Calculate the vertical field of view from sensor geometry.
4. Project a 3D camera-frame point onto the image using the perspective projection equation.
5. Verify your answers with NumPy.

---

## What you need

- Python 3
- NumPy
- Matplotlib

No ROS, no camera hardware — just maths and Python.

# 1. Background

## 1.1 From 3D world to 2D pixel

A camera performs a **perspective projection**: it maps a 3D scene point onto a 2D image plane.

The full pipeline has two steps:

```text
World point P_W  →  [R | t]  →  Camera point P_C  →  K  →  Pixel (u, v)
```

- **[R | t]** is the *extrinsic* matrix: it describes where the camera is in the world (rotation + translation).
- **K** is the *intrinsic* (calibration) matrix: it describes the camera's internal geometry.

The combined equation is:

$$
\lambda \begin{bmatrix} u \\ v \\ 1 \end{bmatrix}
= K \begin{bmatrix} R & t \end{bmatrix}
\begin{bmatrix} X_W \\ Y_W \\ Z_W \\ 1 \end{bmatrix}
$$

where $\lambda = Z_C$ is the depth (a scaling factor that appears because we divide by depth).

## 1.2 The calibration matrix K

$$
K = \begin{bmatrix} a_u & 0 & u_0 \\ 0 & a_v & v_0 \\ 0 & 0 & 1 \end{bmatrix}
$$

| Symbol | Meaning |
|---|---|
| $a_u = k_u f$ | Focal length in pixels, horizontal direction |
| $a_v = k_v f$ | Focal length in pixels, vertical direction |
| $u_0$ | Horizontal pixel coordinate of the principal point (optical center) |
| $v_0$ | Vertical pixel coordinate of the principal point |
| $k_u$, $k_v$ | Pixel density (pixels per mm), i.e. the inverse of pixel pitch |
| $f$ | Focal length in mm |

## 1.3 Focal length in pixels

The pixel-pitch $p$ is the physical distance between adjacent pixel centres (in mm).

$$
f_{\text{pixels}} = \frac{f_{\text{mm}}}{p}
$$

## 1.4 Field of view

The horizontal field of view is related to the sensor width $W$ and focal length $f$ by:

$$
\text{FoV}_H = 2 \arctan\left(\frac{W}{2f}\right)
$$

The same formula applies vertically with sensor height $H$.

# 2. The exercise

You are given a camera with the following specifications:

| Parameter | Value |
|---|---|
| Lens horizontal FoV | 90° |
| Sensor size | Full-frame: 36 mm × 24 mm |
| Resolution | 6 MP |
| Pixel size (pitch) | 12 µm (square pixels, so $k_u = k_v$) |

You will solve three sub-problems:

1. Determine the calibration matrix **K**.
2. Compute the vertical FoV.
3. Project the camera-frame point $P_C = [1, 1, 2]^T$ onto the image.

> **Tip:** work through each step on paper first, then verify with the code cells below.

# 3. Import libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print("NumPy version:", np.__version__)

# 4. Physical sensor parameters

Run this cell as-is. These are the given values for the camera.

In [ ]:
# --- Given sensor parameters ---
sensor_width_mm  = 36.0        # mm (full-frame, horizontal)
sensor_height_mm = 24.0        # mm (full-frame, vertical)
fov_h_deg        = 90.0        # degrees, horizontal field of view
pixel_pitch_um   = 12.0        # µm  (micrometres)
resolution_mp    = 6.0         # megapixels (total)

# Convert pixel pitch to mm
pixel_pitch_mm = pixel_pitch_um * 1e-3
print(f"Pixel pitch: {pixel_pitch_mm} mm")

# 5. Step 1 — Determine the calibration matrix K

## 5.1 Compute the image resolution

The sensor is 36 mm × 24 mm with a pixel pitch of 11 µm. Use this to find the number of pixels horizontally and vertically.

$$
N_u = \frac{\text{sensor width}}{\text{pixel pitch}}, \quad N_v = \frac{\text{sensor height}}{\text{pixel pitch}}
$$

> **Question 5.1:** How many pixels wide and tall is the sensor? Does this match the 6 MP specification? Fill in the `???` below.

In [ ]:
# --- Compute image resolution ---
# Replace ??? with your formula

N_u = ???   # number of pixels horizontally
N_v = ???   # number of pixels vertically

print(f"Image width : {N_u:.0f} pixels")
print(f"Image height: {N_v:.0f} pixels")
print(f"Total pixels: {N_u * N_v / 1e6:.2f} MP")

## 5.2 Compute the focal length in mm

The horizontal FoV is 90°. Use the thin-lens / pinhole relationship:

$$
\text{FoV}_H = 2 \arctan\left(\frac{W}{2f}\right)
\quad \Rightarrow \quad
f = \frac{W}{2 \tan\left(\frac{\text{FoV}_H}{2}\right)}
$$

> **Question 5.2:** What is the focal length in mm? Fill in the `???` below.

In [ ]:
# --- Compute focal length in mm ---
fov_h_rad = np.deg2rad(fov_h_deg)

# Replace ??? with your formula
f_mm = ???

print(f"Focal length: {f_mm:.4f} mm")

## 5.3 Compute the focal length in pixels

$$
a_u = k_u \cdot f = \frac{f_{\text{mm}}}{p}, \qquad a_v = k_v \cdot f = \frac{f_{\text{mm}}}{p}
$$

Because pixel pitch is square ($k_u = k_v$), we have $a_u = a_v$.

> **Question 5.3:** What is the focal length in pixels? Fill in the `???` below.

In [ ]:
# --- Focal length in pixels ---
# Replace ??? with your formula
a_u = ???
a_v = ???   # same as a_u for square pixels

print(f"a_u (focal length in pixels, horizontal): {a_u:.2f} px")
print(f"a_v (focal length in pixels, vertical)  : {a_v:.2f} px")

## 5.4 Compute the principal point

For a well-manufactured camera, the optical axis passes through the centre of the sensor:

$$
u_0 = \frac{N_u}{2}, \qquad v_0 = \frac{N_v}{2}
$$

> **Question 5.4:** Fill in the formulas below.

In [ ]:
# --- Principal point ---
# Replace ??? with your formula
u0 = ???
v0 = ???

print(f"Principal point: u0 = {u0:.1f} px,  v0 = {v0:.1f} px")

## 5.5 Assemble the calibration matrix K

> **Question 5.5:** Assemble **K** as a 3×3 NumPy array using the values you computed above.

In [ ]:
# --- Calibration matrix K ---
# Replace ??? with the correct entries
K = np.array([
    [???,   0, ???],
    [  0, ???, ???],
    [  0,   0,   1]
], dtype=float)

print("Calibration matrix K:")
print(K)

# 6. Step 2 — Vertical field of view

Now apply the same FoV formula but using the sensor *height* instead of width:

$$
\text{FoV}_V = 2 \arctan\left(\frac{H}{2f}\right)
$$

> **Question 6.1:** What is the vertical FoV in degrees? Fill in the `???` below.

> **Question 6.2:** Does the result make physical sense? The sensor is 36 × 24 mm, so the vertical FoV should be smaller than the horizontal FoV of 90°. Check this.

In [ ]:
# --- Vertical FoV ---
# Replace ??? with your formula
fov_v_rad = ???
fov_v_deg = np.rad2deg(fov_v_rad)

print(f"Vertical FoV: {fov_v_deg:.2f} degrees")
print(f"Horizontal FoV: {fov_h_deg:.2f} degrees")
print(f"Ratio H/V sensor: {sensor_width_mm}/{sensor_height_mm} = {sensor_width_mm/sensor_height_mm:.2f} (3:2 aspect ratio)")

# 7. Step 3 — Project a 3D point onto the image

We want to find where the camera-frame point $P_C = [1, 1, 2]^T$ appears in the image.

The perspective projection equation (for a point already in the camera frame, so $R = I$, $t = 0$) is:

$$
\lambda \begin{bmatrix} u \\ v \\ 1 \end{bmatrix} = K \begin{bmatrix} X_C \\ Y_C \\ Z_C \end{bmatrix}
$$

which expands to:

$$
u = u_0 + a_u \frac{X_C}{Z_C}, \qquad v = v_0 + a_v \frac{Y_C}{Z_C}
$$

> **Question 7.1:** Compute (u, v) by hand first, then verify with the code below.

> **Question 7.2:** Is the projected pixel inside the image bounds? What are the image bounds in pixels?

In [ ]:
# --- 3D point in camera frame ---
P_C = np.array([1.0, 1.0, 2.0])   # [X_C, Y_C, Z_C]

# --- Project using the full matrix equation ---
# Homogeneous coordinate: append 1
# lambda * [u, v, 1]^T = K @ P_C

# Replace ??? with your formula
projected_h = ???    # K @ P_C  → gives [lambda*u, lambda*v, lambda]

# Divide by the third element to recover (u, v)
u = projected_h[0] / projected_h[2]
v = projected_h[1] / projected_h[2]

print(f"Projected pixel: u = {u:.2f} px,  v = {v:.2f} px")
print(f"Image size: {N_u:.0f} x {N_v:.0f} px")
print(f"Inside image: {0 <= u <= N_u and 0 <= v <= N_v}")

## 7.2 Visualise the projected point

Run the cell below to see where your projected point falls on the image plane.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# Draw the image boundary
rect = mpatches.Rectangle(
    (0, 0), N_u, N_v,
    linewidth=2, edgecolor='steelblue', facecolor='#f0f4f8'
)
ax.add_patch(rect)

# Draw the principal point
ax.plot(u0, v0, 'b+', markersize=14, markeredgewidth=2, label=f'Principal point ({u0:.0f}, {v0:.0f})')

# Draw the projected point
ax.plot(u, v, 'ro', markersize=10, label=f'P_C=[1,1,2] → ({u:.1f}, {v:.1f})')

ax.set_xlim(-100, N_u + 100)
ax.set_ylim(-100, N_v + 100)
ax.invert_yaxis()   # image coordinates: v increases downward
ax.set_xlabel('u (pixels)')
ax.set_ylabel('v (pixels)')
ax.set_title('Projection of $P_C = [1, 1, 2]^T$ onto the image plane')
ax.legend()
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

# 8. Sanity checks

Use the cells below to verify your results against expected values.

In [ ]:
# --- Sanity check: focal length ---
# For a 90° horizontal FoV on a 36 mm sensor:
#   f = 36 / (2 * tan(45°)) = 36 / 2 = 18 mm
expected_f_mm = 18.0
assert abs(f_mm - expected_f_mm) < 0.01, f"Focal length should be {expected_f_mm} mm, got {f_mm:.4f} mm"
print(f"✓ Focal length: {f_mm:.2f} mm (expected {expected_f_mm} mm)")

# --- Sanity check: focal length in pixels ---
# f_pixels = 18 mm / 0.012 mm = 1500 px
expected_a = 18.0 / 0.012
assert abs(a_u - expected_a) < 1.0, f"a_u should be ~{expected_a:.1f} px, got {a_u:.2f} px"
print(f"✓ a_u: {a_u:.2f} px (expected {expected_a:.2f} px)")

# --- Sanity check: image resolution ---
expected_Nu = round(36.0 / 0.012)
expected_Nv = round(24.0 / 0.012)
assert abs(N_u - expected_Nu) < 5, f"N_u should be ~{expected_Nu}, got {N_u}"
assert abs(N_v - expected_Nv) < 5, f"N_v should be ~{expected_Nv}, got {N_v}"
print(f"✓ Resolution: {N_u:.0f} x {N_v:.0f} px")

# --- Sanity check: vertical FoV ---
expected_fov_v = np.rad2deg(2 * np.arctan(24.0 / (2 * 18.0)))
assert abs(fov_v_deg - expected_fov_v) < 0.1, f"Vertical FoV should be {expected_fov_v:.2f}°, got {fov_v_deg:.2f}°"
print(f"✓ Vertical FoV: {fov_v_deg:.2f}° (expected {expected_fov_v:.2f}°)")

print("\nAll checks passed!")

# 9. Experiments

## Experiment A: Change the FoV

Change `fov_h_deg` to 50° (a more typical telephoto lens) and re-run all cells.

Answer:
1. How does the focal length change?
2. How does the projected position of $P_C = [1,1,2]^T$ change?
3. Why does a longer focal length (narrower FoV) make objects appear larger in the image?

## Experiment B: Move the point closer

Change $P_C$ from `[1, 1, 2]` to `[1, 1, 0.5]` (4× closer to the camera).

Answer:
1. How much does the projected pixel move?
2. Is the relationship between depth and pixel displacement linear or non-linear?
3. What happens if $Z_C = 0$? Why is this a problem?

## Experiment C: Off-axis points

Try projecting these three points and mark all of them on the visualisation:

```python
P1 = [0, 0, 2]   # on the optical axis
P2 = [1, 0, 2]   # shifted right
P3 = [0, 1, 2]   # shifted down
```

Answer:
1. Where does P1 project? (Hint: it should be exactly at the principal point.)
2. Does shifting X by +1 m move the pixel left or right in the image?
3. Why does the image have a flipped orientation compared to the scene?


In [ ]:
# --- Experiment C: project multiple points ---

points = {
    'P1 = [0,0,2] (on axis)': np.array([0.0, 0.0, 2.0]),
    'P2 = [1,0,2] (right)':   np.array([1.0, 0.0, 2.0]),
    'P3 = [0,1,2] (down)':    np.array([0.0, 1.0, 2.0]),
}

fig, ax = plt.subplots(figsize=(8, 5))
rect = mpatches.Rectangle((0, 0), N_u, N_v, linewidth=2,
                            edgecolor='steelblue', facecolor='#f0f4f8')
ax.add_patch(rect)

colors = ['red', 'green', 'orange']
for (label, pt), color in zip(points.items(), colors):
    ph = K @ pt
    pu, pv = ph[0] / ph[2], ph[1] / ph[2]
    ax.plot(pu, pv, 'o', color=color, markersize=10, label=f'{label} → ({pu:.0f}, {pv:.0f})')

ax.plot(u0, v0, 'b+', markersize=16, markeredgewidth=2, label='Principal point')

ax.set_xlim(-100, N_u + 100)
ax.set_ylim(-100, N_v + 100)
ax.invert_yaxis()
ax.set_xlabel('u (pixels)')
ax.set_ylabel('v (pixels)')
ax.set_title('Projection of multiple 3D points')
ax.legend(loc='lower right', fontsize=8)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

# 10. Reflection questions

Answer these in a text cell (double-click to edit) or in your lab report.

1. Why does the K matrix have zeros in the off-diagonal positions? What would it mean if there were a non-zero value at position (0,1)?

2. The principal point $(u_0, v_0)$ is assumed to be at the image centre. In practice, camera manufacturing is not perfect. What effect would a misaligned principal point have on a robot trying to estimate the distance to an object?

3. The MIRTE robot uses a camera with a specific K matrix. If you mount a *different* lens on the same sensor body, does K change? Which entries change and which stay the same?

4. A pixel coordinate $(u, v)$ was measured in an image. Explain in words why you *cannot* uniquely determine the 3D position of the corresponding scene point from a single image. What additional information would you need?


# References

- Lecture slides: *Practical Topics in Spatial AI*, Reza Sabzevari, TU Delft AE4ASM527
- Hartley & Zisserman, *Multiple View Geometry in Computer Vision*, Cambridge University Press
- OpenCV camera calibration: https://docs.opencv.org/4.x/dc/dbb/tutorial_py_calibration.html
